In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import numpy as np


In [11]:

# CONFIGURACIÓN
base_dir = ("saved_files/tfg")
csv_targets = ["comb_band1_values.csv", "comb_band2_values.csv", "comb_band3_values.csv",
               "comb_band4_values.csv", "comb_band5_values.csv", "comb_band6_values.csv",
               "comb_band7_values.csv", "comb_band8_values.csv"]
datos_y = "comb_band1_values.csv"	# Se puede sacar el valor de las boyas de aquí porque es el mismo en todos los csv del base_dir


In [3]:

# Modelos con parámetros por defecto
modelos = {
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Linear Regression": LinearRegression(),
    "KNN Regressor": KNeighborsRegressor(n_neighbors=5),
    "MLP Regressor": MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42),
}


In [47]:
X

,comb_band1_values.csv,comb_band2_values.csv,comb_band3_values.csv,comb_band4_values.csv,comb_band5_values.csv,comb_band6_values.csv,comb_band7_values.csv,comb_band8_values.csv
0,0.023253,0.033456,0.050959,0.018172,0.013998,0.003946,0.038589,0.032478
1,0.010877,0.019773,0.029634,0.007449,0.005139,0.001354,0.047589,0.042911
2,0.013660,0.021562,0.030676,0.008561,0.006035,0.001610,0.048011,0.043989
3,0.011367,0.018162,0.024629,0.006483,0.004468,0.001179,0.042989,0.039022
4,0.016610,0.026924,0.035795,0.007569,0.004952,0.001242,0.045900,0.042178
...,...,...,...,...,...,...,...,...
785,0.015625,0.028108,0.053845,0.021336,0.016742,0.004619,0.064578,0.061733
786,0.015810,0.023660,0.031827,0.008917,0.006227,0.001628,0.065944,0.062911
787,0.002820,0.007173,0.015751,0.004346,0.002989,0.000781,0.064367,0.062178
788,0.018086,0.027690,0.036716,0.009686,0.006544,0.001650,0.072011,0.068611


In [39]:

resumen_total = []
# for root, dirs, files in os.walk(base_dir):
#     print(files)
#     print(root)
#     #if "S2-BPA-AllDates" == root.split("/")[-1]:
#     presentes = [f for f in files if f in csv_targets]
#     if all(name in presentes for name in csv_targets):

#for name in os.listdir(base_dir):

datos_por_archivo = {name: [] for name in csv_targets}
valores_reales = []
#print(name)
df_dict = {name: pd.read_csv(os.path.join(base_dir, name)) for name in csv_targets}
for name in csv_targets:
    datos_por_archivo[name].extend(df_dict[name].iloc[:, 3].tolist())
valores_reales.extend(df_dict[datos_y].iloc[:, 2].tolist())

# # Nada si no hay datos suficientes
# if len(datos_por_archivo[name]) < 2:
#     continue

X = pd.DataFrame({name: datos_por_archivo[name] for name in csv_targets})
y = pd.Series(valores_reales, name="BoyaReal")
data = pd.concat([X, y], axis=1).dropna()
X = data[csv_targets]
y = data["BoyaReal"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()

resultados = []


In [40]:
for nombre, modelo in modelos.items():
    print(nombre, modelo)

Random Forest RandomForestRegressor(random_state=42)
Linear Regression LinearRegression()
KNN Regressor KNeighborsRegressor()
MLP Regressor MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42)


In [41]:
for nombre, modelo in modelos.items():
    print()
    if nombre in ["KNN Regressor", "MLP Regressor"]:
        modelo.fit(X_train_scaled, y_train_scaled)
        y_pred = scaler_y.inverse_transform(modelo.predict(X_test_scaled).reshape(-1, 1)).ravel()
    else:
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    nrmse = rmse / (y_test.max() - y_test.min()) if (y_test.max() - y_test.min()) != 0 else -1

    r2_sklearn = r2_score(y_test, y_pred)
    corr = np.corrcoef(y_test, y_pred)[0, 1]
    r2_corr = corr ** 2

    #profundidad = root.split("/")[-2]
    resultados.append(
        {
            "Modelo": nombre,
            "MSE": mse,
            "RMSE": rmse,
            "NRMSE": nrmse,
            "R² sklearn": r2_sklearn,
            "R² correlación": r2_corr,
            #"Profundidad": profundidad
        }
    )

    resumen_total.extend(resultados)


In [27]:
y_test.tolist()

[0.9318,
 0.3392,
 0.3016,
 0.2555,
 0.2827,
 0.3731,
 2.185,
 1.9823,
 0.8198,
 0.225,
 0.1151,
 0.2308,
 0.3213,
 0.4307,
 3.1727,
 18.0257,
 0.3488,
 0.507,
 0.0826,
 0.8554,
 0.947,
 0.1424,
 0.265,
 0.4792,
 0.4792,
 0.4821,
 0.3295,
 0.9212,
 0.4528,
 0.2963,
 0.87,
 1.4325,
 0.9501,
 1.0737,
 0.4446,
 0.4122,
 0.1892,
 0.3365,
 0.7706,
 1.033,
 0.2601,
 0.3468,
 2.3132,
 0.8111,
 0.2593,
 0.686,
 0.4262,
 0.328,
 0.1845,
 0.2924,
 2.3446,
 0.1153,
 0.2883,
 0.3713,
 0.1737,
 0.2714,
 0.3045,
 17.2703,
 0.5209,
 1.3965,
 0.1899,
 0.8184,
 0.3462,
 0.4143,
 0.4193,
 0.5733,
 0.8145,
 1.3604,
 0.2175,
 0.7222,
 0.3669,
 0.8697,
 0.6572,
 0.873,
 0.295,
 1.8043,
 0.1859,
 0.206,
 0.2516,
 2.5606,
 0.1587,
 1.2307,
 0.5343,
 1.8445,
 1.0288,
 0.3257,
 2.4923,
 1.1628,
 1.1658,
 0.3196,
 0.7905,
 2.8386,
 0.2053,
 0.3276,
 0.5752,
 0.4247,
 0.1581,
 1.8545,
 0.9208,
 0.3568,
 0.2926,
 0.72,
 0.6433,
 0.1603,
 18.1624,
 0.0443,
 0.6209,
 0.4647,
 0.3693,
 0.0624,
 1.8967,
 0.8716,
 0.2

In [25]:
y_pred

array([ 0.55456558,  0.57257223,  0.60576489,  0.63595374,  0.70228468,
        0.61881632,  1.27795811,  1.49778456,  1.44789679,  0.56223416,
        0.77541054,  0.5232659 ,  0.63673659,  1.10270261,  2.24458995,
       16.62902569,  0.83240733,  0.61037802,  0.97810656,  0.60024877,
        1.2077368 ,  0.98997026,  0.46934842,  1.54113955,  0.73348253,
        0.68120002,  0.57957538,  1.30348259,  0.68675965,  0.49661528,
        0.94213133,  1.85491656,  1.07350612,  0.85693788,  1.45577602,
        3.33698276,  0.27399679,  1.21963477,  1.15912886,  0.6436694 ,
        1.2554906 ,  0.68366877,  2.05437961,  1.47628149,  1.07844821,
        0.47785351,  0.59634278,  1.4215988 ,  1.22988405,  0.43704811,
        2.71605202,  0.75502483,  1.04014445,  0.47163668,  0.57179106,
        0.82522105,  0.43732767, 15.48448126,  0.79124895,  0.64459326,
        0.46910217,  0.73643987,  0.76908128,  0.67679412,  0.97328313,
        0.61437379,  1.830665  ,  1.79740272,  1.19667133,  0.88

In [42]:
resumen_total

[{'Modelo': 'Random Forest',
  'MSE': 3.076475636994065,
  'RMSE': 1.753988493974252,
  'NRMSE': 0.096808633023013,
  'R² sklearn': 0.6407987717228436,
  'R² correlación': 0.647254394834646},
 {'Modelo': 'Random Forest',
  'MSE': 3.076475636994065,
  'RMSE': 1.753988493974252,
  'NRMSE': 0.096808633023013,
  'R² sklearn': 0.6407987717228436,
  'R² correlación': 0.647254394834646},
 {'Modelo': 'Linear Regression',
  'MSE': 4.60547941175563,
  'RMSE': 2.1460380732306756,
  'NRMSE': 0.1184471922127969,
  'R² sklearn': 0.4622763003174176,
  'R² correlación': 0.5650973773908624},
 {'Modelo': 'Random Forest',
  'MSE': 3.076475636994065,
  'RMSE': 1.753988493974252,
  'NRMSE': 0.096808633023013,
  'R² sklearn': 0.6407987717228436,
  'R² correlación': 0.647254394834646},
 {'Modelo': 'Linear Regression',
  'MSE': 4.60547941175563,
  'RMSE': 2.1460380732306756,
  'NRMSE': 0.1184471922127969,
  'R² sklearn': 0.4622763003174176,
  'R² correlación': 0.5650973773908624},
 {'Modelo': 'KNN Regressor',

In [ ]:

# Exportar a CSV
resumen_df = pd.DataFrame(resumen_total).sort_values(by=["Profundidad", "Modelo"], ascending=[True, False])
resumen_df.to_csv("resumen_total_ML.csv", index=False)
